In [0]:
# --- Monta a pasta do app dentro do projeto (com plano B na pasta do usuário) ---
import shutil
import joblib
from pathlib import Path
from databricks.sdk import WorkspaceClient

origem = Path("/Workspace/Users/mateus@asn.rocks/Hackathon_Terra_Signal")

def montar_pasta_do_app(destino):
    destino.mkdir(parents=True, exist_ok=True)
    for arquivo in ["app.py", "requirements.txt", "inference.csv"]:
        shutil.copy(origem / arquivo, destino / arquivo)
    artefato = joblib.load(origem / "modelo_churn.joblib")
    joblib.dump(artefato, destino / "modelo_churn.joblib", compress=3)  # abaixo de 10 MB
    (destino / "app.yaml").write_text('command: ["streamlit", "run", "app.py"]\n')
    return destino

try:
    pasta_app = montar_pasta_do_app(origem / "app")
except Exception as erro:
    print(f"[AVISO] Não deu para gravar na pasta do projeto ({erro}). Usando a pasta do usuário.")
    usuario = WorkspaceClient().current_user.me().user_name
    pasta_app = montar_pasta_do_app(Path(f"/Workspace/Users/{usuario}/terrasignal_app"))

print(f"[INFO] Pasta do app: {pasta_app}")
for f in sorted(pasta_app.iterdir()):
    print(f"{f.name}: {f.stat().st_size / 1024 / 1024:.2f} MB")